<a href="https://colab.research.google.com/github/marsya505/DataMining/blob/main/Week13-Iris.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load dataset Iris
iris = load_iris()
X = iris.data
y = iris.target

# Bagi data menjadi data pelatihan dan data uji
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standarisasi fitur
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Inisialisasi dan pelatihan model MLPClassifier
mlp = MLPClassifier(hidden_layer_sizes=(2,), max_iter=5000, alpha=0.01, solver='adam', random_state=42)
mlp.fit(X_train, y_train)

# Prediksi pada data uji
y_pred = mlp.predict(X_test)

# Evaluasi model
accuracy = accuracy_score(y_test, y_pred)
print(f'Akurasi: {accuracy * 100}%')
print(classification_report(y_test, y_pred, target_names=iris.target_names))

# Menampilkan arsitektur jaringan syaraf
print("\nArsitektur Neural Network:")
print(f"Input Layer: {X_train.shape[1]} neurons")
for i, size in enumerate(mlp.hidden_layer_sizes):
    print(f"Hidden Layer {i + 1}: {size} neurons")
print(f"Output Layer: {mlp.n_outputs_} neurons")
print(f"Fungsi Aktivasi: {mlp.activation}")

# Jika ingin melihat bobot dan bias hasil pelatihan
print("\nBobot dan Bias:")
print("Bobot antara input dan hidden layer:")
print(mlp.coefs_[0])
print("Bobot antara hidden dan output layer:")
print(mlp.coefs_[1])
print("Bias dari hidden layer:")
print(mlp.intercepts_[0])
print("Bias dari output layer:")
print(mlp.intercepts_[1])

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score

# Load dataset Iris
iris = load_iris()

X = iris.data
y = iris.target

# One-hot encoding untuk label
encoder = OneHotEncoder(sparse_output=False)
y = encoder.fit_transform(y.reshape(-1, 1))

# Bagi data menjadi data pelatihan dan data uji
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fungsi aktivasi sigmoid
def sigmoid(x):
  return 1 / (1 + np.exp(-x))

# Turunan fungsi sigmoid
def sigmoid_derivative(x):
  return x * (1 - x)

# Parameter jaringan neural
input_layer_neurons = X_train.shape[1]
hidden_layer_neurons = 10
output_layer_neurons = y_train.shape[1]

# Inisialisasi bobot dan bias
np.random.seed(42)
weights_input_hidden = np.random.uniform(size=(input_layer_neurons, hidden_layer_neurons))
weights_hidden_output = np.random.uniform(size=(hidden_layer_neurons, output_layer_neurons))
bias_hidden = np.random.uniform(size=(1, hidden_layer_neurons))
bias_output = np.random.uniform(size=(1, output_layer_neurons))

# Hiperparameter
learning_rate = 0.01
epochs = 5000

# Proses pelatihan
for epoch in range(epochs):
  # Forward propagation
  hidden_layer_input = np.dot(X_train, weights_input_hidden) + bias_hidden
  hidden_layer_output = sigmoid(hidden_layer_input)

  output_layer_input = np.dot(hidden_layer_output, weights_hidden_output) + bias_output
  predicted_output = sigmoid(output_layer_input)

  # Backpropagation
  error = y_train - predicted_output
  d_predicted_output = error * sigmoid_derivative(predicted_output)
  error_hidden_layer = d_predicted_output.dot(weights_hidden_output.T)
  d_hidden_layer = error_hidden_layer * sigmoid_derivative(hidden_layer_output)

  # Update bobot dan bias
  weights_hidden_output += hidden_layer_output.T.dot(d_predicted_output) * learning_rate
  bias_output += np.sum(d_predicted_output, axis=0, keepdims=True) * learning_rate
  weights_input_hidden += X_train.T.dot(d_hidden_layer) * learning_rate
  bias_hidden += np.sum(d_hidden_layer, axis=0, keepdims=True) * learning_rate

  # Cetak error setiap 1000 epochs
  if (epoch + 1) % 1000 == 0:
    loss = np.mean(np.square(error))
    print(f'Epoch {epoch + 1}, Loss: {loss}')

# Prediksi pada data uji
hidden_layer_input = np.dot(X_test, weights_input_hidden) + bias_hidden
hidden_layer_output = sigmoid(hidden_layer_input)
output_layer_input = np.dot(hidden_layer_output, weights_hidden_output) + bias_output
predicted_output = sigmoid(output_layer_input)

# Evaluasi model
y_test_labels = np.argmax(y_test, axis=1)
predicted_labels = np.argmax(predicted_output, axis=1)
accuracy = accuracy_score(y_test_labels, predicted_labels)
print(f'Accuracy: {accuracy * 100}%')